In [1]:
import torch
from d2l import torch as d2l
import torch.nn as nn

In [2]:
import numpy
import math
import time

In [ ]:
class benchmark:
    def __init__(self,description):
        self.name=description
        return
    def __enter__(self):
        self.start_time=time.time()
    def __exit__(self,*args):
        print(f"{self.name}:duration{time.time()-self.start_time:.3f}s")


net=nn.Sequential(nn.Linear(10,12),nn.ReLU(),
                  nn.Linear(12,8),nn.ReLU(),
                  nn.Linear(8,1),nn.ReLU()).to(torch.device('cuda:2'))

net0=torch.jit.script(net)

#编译后运算
with benchmark("no script"):
    for i in range(10000):
        x=torch.rand(1024,10).to(torch.device('cuda:2'))
        y=net(x)

with benchmark("script"):
    for i in range(10000):
        x=torch.rand(1024,10).to(torch.device('cuda:2'))
        y=net0(x)

/nfs3/wangfengyu/conda_envs/mytorch/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


no script:duration2.396s
script:duration2.174s


In [ ]:
#设备预热：GPU在首次调用时计算time可能会有偏差。
for i in range(10):
    x=torch.randn([1024,1024]).to(torch.device('cuda:2'))
    y=torch.normal(0,1,[1024,1024]).to(torch.device('cuda:2'))
    torch.mm(x,y)

torch.cuda.synchronize(torch.device('cuda:2'))   #CUDA默认CPU与GPU之间是并行的，CPU发送完GPU指令可以继续执行计算，synchronize会要求等待该设备的计算完成后才继续进行下一步cpu的执行

#CPU派发指令
with benchmark("1 script"):
    for i in range(10):
        x=torch.randn([1024,1024]).to(torch.device('cuda:2'))
        y=torch.normal(0,1,[1024,1024]).to(torch.device('cuda:2'))
        mm+=torch.mm(x,y)

#GPU运算
with benchmark("2 script"):
    for i in range(10):
        x=torch.randn([1024,1024]).to(torch.device('cuda:2'))
        y=torch.normal(0,1,[1024,1024]).to(torch.device('cuda:2'))
        mm+=torch.mm(x,y)
    torch.cuda.synchronize(torch.device('cuda:2'))    #如果不加synchronize，测出的实际是CPU下发完指令的时间。

mm=torch.zeros([1024,1024]).to(torch.device('cuda:2'))

#GPU运算
with benchmark("3 script"):
    for i in range(10):
        x=torch.randn([1024,1024]).to(torch.device('cuda:2'))
        y=torch.normal(0,1,[1024,1024]).to(torch.device('cuda:2'))
        mm+=torch.mm(x,y)
    torch.cuda.synchronize(torch.device('cuda:2'))    #如果GPU的执行结果需要被使用，则无法实现cpu与gpu的异步，即在mm+=处，cpu会等待gpu完成。

with benchmark("4 script"):
    for i in range(10):
        x=numpy.random.normal(size=(1024,1024))
        y=numpy.random.normal(size=(1024,1024))
        numpy.dot(x,y)

no script:duration0.149s
script:duration0.855s


In [ ]:
torch.cuda.synchronize(torch.device('cuda:1'))  
torch.cuda.synchronize(torch.device('cuda:2'))  

devices = d2l.try_all_gpus() 
def run(x):  
    return [x.mm(x) for _ in range(50)] 
x_gpu1 = torch.rand(size=(4000, 4000), device=devices[0]) 
x_gpu2 = torch.rand(size=(4000, 4000), device=devices[1])

run(x_gpu1) 
run(x_gpu2) # 预热设备 
torch.cuda.synchronize(devices[0]) 
torch.cuda.synchronize(devices[1]) 
with d2l.Benchmark('GPU1 time'):
  run(x_gpu1) 
  torch.cuda.synchronize(devices[0])
  
with d2l.Benchmark('GPU2 time'):
    run(x_gpu2)
    torch.cuda.synchronize(devices[1])

with d2l.Benchmark('GPU1 & GPU2'):
      run(x_gpu1)
      run(x_gpu2)
      torch.cuda.synchronize()

GPU1 time: 0.5601 sec
GPU2 time: 0.6832 sec
GPU1 & GPU2: 0.6710 sec
